# NB59: IoT Aggregation

Windowed aggregation of IoT metrics stored in Cassandra.

## 1. Environment Setup

This cell installs **Java 8**, **Spark 3.5.0**, **Kafka 3.6.1**, and necessary Python libraries (`pyspark`, `kafka-python`, `redis`, `pymongo`, `elasticsearch`, `cassandra-driver`, `minio`). It also sets environment variables for Java and Spark.

In [ ]:
# Install Dependencies
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!wget -q https://archive.apache.org/dist/kafka/3.6.1/kafka_2.13-3.6.1.tgz
!tar xf kafka_2.13-3.6.1.tgz
!pip install -q findspark pyspark kafka-python redis pymongo elasticsearch==7.10.1 cassandra-driver minio "numpy<2.0.0"

# Environment Variables
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"
import findspark
findspark.init()

## 2. Start Services

This cell starts the required distributed services in the background:
- **Kafka & Zookeeper**: Event streaming platform.
- **Cassandra**: Wide-column store.

In [ ]:
# Start Kafka
!!./kafka_2.13-3.6.1/bin/zookeeper-server-start.sh -daemon ./kafka_2.13-3.6.1/config/zookeeper.properties
!!./kafka_2.13-3.6.1/bin/kafka-server-start.sh -daemon ./kafka_2.13-3.6.1/config/server.properties
# Start Cassandra
!echo "deb http://www.apache.org/dist/cassandra/debian 40x main" | tee -a /etc/apt/sources.list.d/cassandra.sources.list
!curl https://downloads.apache.org/cassandra/KEYS | apt-key add -
!apt-get update -qq > /dev/null
!apt-get install cassandra -qq > /dev/null
!service cassandra start

import time
time.sleep(30) # Wait for startup

## 3. Create Kafka Topic

Creates a topic named `input-topic` with 1 partition and replication factor 1.

In [ ]:
# Create Topic
!./kafka_2.13-3.6.1/bin/kafka-topics.sh --create --topic input-topic --bootstrap-server localhost:9092 --replication-factor 1 --partitions 1

## 4. Producer

Sends sensor events.

In [ ]:
from kafka import KafkaProducer
import json, time, random
print("Starting IoT Producer...")
producer = KafkaProducer(bootstrap_servers='localhost:9092')
print("Sending 500 events...")
for i in range(500):
    producer.send('input-topic', json.dumps({'id': 's1', 'val': i}).encode('utf-8'))
producer.flush()
print("Producer finished.")

## 5. Spark Aggregation -> Cassandra

Counts events in the batch and updates a counter in Cassandra.

In [ ]:
%%writefile kafka_consumer.py
from pyspark.sql import SparkSession
from cassandra.cluster import Cluster

# Init Keyspace
cluster = Cluster(['127.0.0.1'])
session = cluster.connect()
session.execute("CREATE KEYSPACE IF NOT EXISTS iot WITH replication = {'class': 'SimpleStrategy', 'replication_factor': 1}")
session.execute("CREATE TABLE IF NOT EXISTS iot.aggs (id text PRIMARY KEY, count int)")
session.shutdown()

spark = SparkSession.builder.appName("IoT").getOrCreate()

def process_batch(df, epoch_id):
    count = df.count()
    if count > 0:
        cluster = Cluster(['127.0.0.1'])
        session = cluster.connect('iot')
        session.execute(f"INSERT INTO aggs (id, count) VALUES ('s1', {count})")
        session.shutdown()
        print(f"Batch {epoch_id}: Updates sent to Cassandra.")

print("Starting Spark Streaming Job...")
df = spark.readStream.format("kafka").option("kafka.bootstrap.servers", "localhost:9092").option("subscribe", "input-topic").load()
query = df.selectExpr("CAST(value AS STRING)").writeStream.foreachBatch(process_batch).start()
query.awaitTermination(30)
print("Spark Job Finished.")

In [ ]:
!spark-submit --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0 kafka_consumer.py

## 6. Verification

Check accumulated count in Cassandra.

In [ ]:
from cassandra.cluster import Cluster
cluster = Cluster(['127.0.0.1'])
session = cluster.connect('iot')
print(session.execute("SELECT * FROM aggs").one())